# Phase 1.5 — causal operation-specialization battery
selectivity는 raw-Q ceiling(0.98)에 confound됨 → expert가 operation-specific 연산을 *인과적으로* 하는지 검증.
- **SWAP**: op-X 질문을 op-Y signature로 라우팅. 대각(X|X)≫비대각(X|Y) = operation-bound (positive).
- **LESION**: op-X experts 제거 → 대각 우세(X-질문만 타격) = specific.
둘 다 null(무관/균일)이면 → experts 교환가능 = 표면 tagging (negative).

In [ ]:
# --- setup ---
import os, sys
from pathlib import Path
import numpy as np, torch
BASE = Path('/content/drive/MyDrive/sideproject')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from phase1_5.data import MCCorpusConfig, build_mc_corpus, encode_or_load_mc, make_mc_loaders, MODE_Q_ONLY
cfg = MCCorpusConfig(cache_root=str(BASE / 'out' / 'phase1_5' / 'cache'))
print('device', device)

In [ ]:
# --- train top-k model (K_active = 4) ---
from phase1_5.model import Phase15MoE, MOD_KG_HYPERNET
from phase1_5.train import TrainConfig, train_phase15
corpus = build_mc_corpus(cfg)
data = encode_or_load_mc(corpus, cfg, encoding_mode=MODE_Q_ONLY, device=device)
tr, va, te = make_mc_loaders(data, batch_size=64)
model = Phase15MoE(d_emb=data['q_tokens'].shape[-1], d_z=256, k_routed=64,
                   modulation=MOD_KG_HYPERNET, lb_strategy='aux_free',
                   lb_target_active=4.0, routing='topk')
tc = TrainConfig(epochs=40, lr=1e-3, k_target=4.0, lam_z=1e-3, use_best_val=False, seed=0)
res = train_phase15(model, tr, val_loader=va, cfg=tc, device=device, progress=True)
model = res['model']
print('final val_acc:', (res['history'] or [{}])[-1].get('val_mc_acc'))

In [ ]:
# --- build probe batch (test split, regex op-labels, drop other/rare) ---
from phase1_5.eval import build_operation_labels
test_mask = data['split'] == 'test'
questions = corpus[corpus['split'] == 'test']['question'].tolist()
labels, keep = build_operation_labels(questions, min_count=20)

def _sub(arr):
    return arr[test_mask][keep]

def _t(x):
    return torch.from_numpy(np.asarray(x)).float()

batch = {
    'q_tokens': _t(_sub(data['q_tokens'])), 'q_mask': _t(_sub(data['q_mask'])),
    'p_tokens': _t(_sub(data['p_tokens'])), 'p_mask': _t(_sub(data['p_mask'])),
    'cand_pooled': _t(_sub(data['cand_pooled'])),
    'answer_idx': torch.from_numpy(_sub(data['answer_idx']).astype('int64')),
}
op_labels = labels
print('probe n =', len(op_labels), '| ops =', sorted(set(op_labels.tolist())))

In [ ]:
# --- run battery: SWAP + LESION ---
from phase1_5.intervention import operation_signature, lesion_specificity, operation_swap
sigs, tops = operation_signature(model, batch['q_tokens'], batch['q_mask'], op_labels, k_top=4, device=device)
swap = operation_swap(model, batch, op_labels, sigs, device=device)
lesion = lesion_specificity(model, batch, op_labels, tops, device=device)
ops = sorted(sigs)

def _matrix(title, getter):
    print(title)
    print('             ' + ' '.join(f'{o[:6]:>7}' for o in ops))
    for x in ops:
        print(f'{x[:11]:>11}  ' + ' '.join(f'{getter(x, y):7.3f}' for y in ops))

_matrix('SWAP acc[X-questions | Y-signature]  (\u2192 \ub300\uac01 X|X \uc6b0\uc138 = operation-bound):',
        lambda x, y: swap[x][y])
print()
_matrix('LESION drop[X-experts removed | Y-questions]  (\u2192 \ub300\uac01 \uc6b0\uc138 = specific):',
        lambda x, y: lesion['drop'][x][y])
print('\nbaseline acc:', {o: round(lesion['baseline'][o], 3) for o in ops})
diag = np.mean([swap[x][x] for x in ops])
offd = np.mean([swap[x][y] for x in ops for y in ops if x != y])
print(f'\nSWAP diagonal mean {diag:.3f} vs off-diagonal mean {offd:.3f}  (diag\u226boff \u2192 positive)')

In [ ]:
# --- specialization battery across routing/LB (GENERALIZING models, Phase 1a) ---
# Run after cell 3 (needs batch, op_labels, tr, va, data).
# relu_l1 = router LEARNS k_active (emergent); topk = forced. Regularized
# (weight_decay 1e-1, epochs 20, best_val) to remove the memorization confound:
# the prior un-regularized run hit train_acc ~100% / val_loss 1.2->12, so its
# specialization numbers were on memorized models. The 'gap' column
# (train_acc - val_acc at the evaluated best-val checkpoint) confirms
# generalization; battery (col_spec/swap_match) is only trustworthy when gap is SMALL.
from phase1_5.model import Phase15MoE, MOD_KG_HYPERNET
from phase1_5.train import TrainConfig, train_phase15
from phase1_5.intervention import operation_signature, lesion_specificity, operation_swap


def col_spec(drop, ops):   # lesion: removing Y's own experts hurts Y-questions most? (column-wise, rigorous)
    return float(np.mean([drop[y][y] - np.mean([drop[x][y] for x in ops if x != y]) for y in ops]))


def row_match(acc, ops):   # swap: fraction of ops whose own signature is best for own questions (uncontrolled)
    return float(np.mean([acc[x][x] >= max(acc[x].values()) - 1e-9 for x in ops]))


ops = sorted(set(op_labels.tolist()))
# (routing, lb_strategy, k_active_target, k_top_for_battery, lam_z)
configs = [
    ('relu_l1', 'off',      4.0, 8, 0.0),    # ka emergent, unsuppressed
    ('relu_l1', 'aux_free', 4.0, 8, 0.0),
    ('topk',    'off',      4.0, 4, 1e-3),
    ('topk',    'aux_free', 4.0, 4, 1e-3),
]
print('routing   lb        val_acc  gap    K_act  col_spec  swap_match   (chance=%.2f)' % (1.0 / len(ops)))
for routing, lb, kat, ktop, lz in configs:
    m = Phase15MoE(d_emb=data['q_tokens'].shape[-1], d_z=256, k_routed=64,
                   modulation=MOD_KG_HYPERNET, lb_strategy=lb,
                   lb_target_active=kat, routing=routing)
    r = train_phase15(m, tr, val_loader=va,
                      cfg=TrainConfig(epochs=20, lr=1e-3, k_target=kat, lam_z=lz,
                                      weight_decay=1e-1, use_best_val=True, seed=0),
                      device=device, progress=False)
    m = r['model']
    hist = r['history']
    be = hist[int(np.argmin([e.get('val_loss', 1e9) for e in hist]))]   # best-val checkpoint (= the model evaluated)
    val_acc = be.get('val_mc_acc') or 0
    gap = (be.get('mc_acc') or 0) - val_acc
    k_act = be.get('k_active_mean') or 0
    sigs, tops = operation_signature(m, batch['q_tokens'], batch['q_mask'], op_labels, k_top=ktop, device=device)
    sw = operation_swap(m, batch, op_labels, sigs, device=device)
    dr = lesion_specificity(m, batch, op_labels, tops, device=device)['drop']
    print('%-9s %-9s %.3f   %+.2f  %5.1f  %+.3f      %.2f' % (
        routing, lb, val_acc, gap, k_act, col_spec(dr, ops), row_match(sw, ops)))
print('\ngap small (generalizing) -> trust battery.  col_spec~0 -> NEGATIVE(pivot);  col_spec>0 -> specialized(proceed)')


In [ ]:
# --- regularized re-measurement: does the weak signal (col_spec ~0.05) STRENGTHEN? ---
# Self-contained: run after cells 1,2,3 (needs data, tr, va, batch, op_labels). No
# need to run the slow matrix cell. overfit is the only remaining lever (data
# expansion blocked: LSAT-LR is 5-choice / too small). Test: dropout + model-shrink
# -> does the topk+aux_free signal grow & sharpen vs baseline col_spec ~0.05?
import numpy as np
from phase1_5.model import Phase15MoE, MOD_KG_HYPERNET
from phase1_5.train import TrainConfig, train_phase15
from phase1_5.intervention import operation_signature, lesion_specificity, operation_swap


def col_spec(drop, ops):   # lesion column-wise: Y's own experts hurt Y-questions most? (rigorous)
    return float(np.mean([drop[y][y] - np.mean([drop[x][y] for x in ops if x != y]) for y in ops]))


def row_match(acc, ops):   # swap: fraction of ops whose own signature is best for own questions
    return float(np.mean([acc[x][x] >= max(acc[x].values()) - 1e-9 for x in ops]))


ops = sorted(set(op_labels.tolist()))
reg_configs = [
    ('full+drop0.3',   dict(d_hidden_expert=512, k_routed=64, dropout=0.3)),
    ('shrink+drop0.3', dict(d_hidden_expert=128, k_routed=32, dropout=0.3)),
]
print('config           seed  val_acc  gap     col_spec  swap_match   (baseline col_spec~0.05)')
for name, mk in reg_configs:
    for seed in [0, 1, 2]:
        m = Phase15MoE(d_emb=data['q_tokens'].shape[-1], d_z=256,
                       modulation=MOD_KG_HYPERNET, lb_strategy='aux_free',
                       lb_target_active=4.0, routing='topk', **mk)
        r = train_phase15(m, tr, val_loader=va,
                          cfg=TrainConfig(epochs=20, lr=1e-3, k_target=4.0, lam_z=1e-3,
                                          weight_decay=1e-1, use_best_val=True, seed=seed),
                          device=device, progress=False)
        m = r['model']
        hist = r['history']
        be = hist[int(np.argmin([e.get('val_loss', 1e9) for e in hist]))]
        va_ = be.get('val_mc_acc') or 0
        gap = (be.get('mc_acc') or 0) - va_
        sigs, tops = operation_signature(m, batch['q_tokens'], batch['q_mask'], op_labels, k_top=4, device=device)
        sw = operation_swap(m, batch, op_labels, sigs, device=device)
        dr = lesion_specificity(m, batch, op_labels, tops, device=device)['drop']
        print('%-15s  %d    %.3f   %+.2f   %+.3f     %.2f' % (
            name, seed, va_, gap, col_spec(dr, ops), row_match(sw, ops)))
print('\ncol_spec grows (>0.10) across seeds + diagonal sharper -> real seed (PROCEED 1b).')
print('flat ~0.05 regardless of regularization/shrink -> weak ceiling (reframe / pivot).')
